# Tokenizer training

In [2]:
import glob
from tokenizers import decoders
import os
from tqdm import tqdm
import time
from tokenizers import SentencePieceBPETokenizer, BertWordPieceTokenizer, ByteLevelBPETokenizer, CharBPETokenizer
from transformers import PreTrainedTokenizerFast
import datetime
import pandas as pd
from tokenizers.normalizers import Replace

In [3]:
os.getcwd()

'c:\\Users\\91942\\Desktop\\Kishan_NLP'

In [4]:
!unzip all_final_datasets.zip

'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
# Define parameters (instead of using argparse)
vocab_size = 50000  # You can modify this value directly in the notebook
file_paths = glob.glob("./sangrah.csv")
save_path = "./sangrah_tokenizers/"
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Define special tokens for beginning and end of sentence
bos_tok = "<bos>"
eos_tok = "<eos>"

# Define extra characters, removing digits and symbols not used in Nepali

extra_char = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "०", "१", "२", "३", "४", "५", "६", "७", "८", "९"]

"""
'ऀ', 'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ॠ', 'ऌ', 'ॡ', 'ए', 'ऐ', 'ओ', 'औ', 'अं', 'अः',  # Vowels
    'क', 'ख', 'ग', 'घ', 'ङ', 'च', 'छ', 'ज', 'झ', 'ञ', 'ट', 'ठ', 'ड', 'ढ', 'ण', 'त', 'थ', 'द', 'ध', 'न',
    'प', 'फ', 'ब', 'भ', 'म', 'य', 'र', 'ल', 'व', 'श', 'ष', 'स', 'ह', 'ळ', 'क्ष', 'ज्ञ',  # Consonants
    'ॐ', 'ॖ', 'ॗ', 'क़', 'ख़', 'ग़', 'ज़', 'ड़', 'ढ़', 'फ़', 'य़',
    'ॢ', 'ॣ', '०', '१', '२', '३', '४', '५', '६', '७', '८', '९',  # Numbers & additional symbols
    '।', '॥', 'ं', 'ः', 'ॆ', 'े', 'ै', 'ॉ', 'ॊ', 'ो', 'ौ', '्', 'ॎ', 'ॏ', '॑', '॒', '॓', '॔', 'ॕ'
"""
def train_tokenizer(file_paths, tokenizers, vocab_size=49152):                # change_this
    st = time.time()
    print(f'Vocab size: {vocab_size}')
    print("Files for training:", file_paths)

    # Iterate through files
    for j, tokenizer in enumerate(tokenizers):
      for i, file_name in enumerate(file_paths):
            print(f"Processing file: {file_name}, {tokenizer.__class__.__name__}")
            try:
                df = pd.read_csv(file_name)
                tokenizer.train_from_iterator(
                    df["text"].tolist(),
                    vocab_size = vocab_size,
                    min_frequency = 5,
                    special_tokens=["<pad>", "<cls>", "<sep>", "<mask>", "<unk>", bos_tok, eos_tok, "<user>", "<assistant>"] + extra_char,
                    show_progress=True
                )
                tokenizer.save_model(save_path , f"{file_name}_{tokenizer.__class__.__name__}_{vocab_size}")

                # Convert to Hugging Face-compatible tokenizer
                transformer_tokenizer = PreTrainedTokenizerFast(
                    tokenizer_object=tokenizer,
                    bos_token=bos_tok,
                    eos_token=eos_tok,
                    unk_token="<unk>",
                    pad_token="<pad>",
                    mask_token="<mask>",
                    padding_side="left",
                    truncation_side="right",
                    additional_special_tokens = ["<user>", "<assistant>"],
                    clean_up_tokenization_spaces=False,
                )

                transformer_tokenizer.backend_tokenizer.normalizer = Replace(' ', '▁')
                transformer_tokenizer.backend_tokenizer.pre_tokenizer = None
                transformer_tokenizer.backend_tokenizer.decoder = decoders.Replace('▁', ' ')

                # Save Hugging Face-compatible tokenizer
                transformer_tokenizer.save_pretrained(save_path + f"{file_name}_{tokenizer.__class__.__name__}_{vocab_size}_transformer")
                print(f"Saved tokenizer to {file_name}_{tokenizer.__class__.__name__}_{vocab_size} in {(time.time() - st) / 60:.2f} minutes")

            except Exception as e:
                print("Error in training tokenizer:", e)
                print("Trained for {} minutes".format((time.time() - st) / 60))

            print("Completed tokenizing file:", file_name, tokenizer.__class__.__name__)

In [6]:
tokenizers = [SentencePieceBPETokenizer()] # BertWordPieceTokenizer(),  SentencePieceBPETokenizer() , BertWordPieceTokenizer(). CharBPETokenizer(), ByteLevelBPETokenizer()

# Run the tokenizer training function
train_tokenizer(file_paths, tokenizers, vocab_size=vocab_size)

Vocab size: 50000
Files for training: ['./sangrah.csv']
Processing file: ./sangrah.csv, SentencePieceBPETokenizer
Saved tokenizer to ./sangrah.csv_SentencePieceBPETokenizer_50000 in 0.59 minutes
Completed tokenizing file: ./sangrah.csv SentencePieceBPETokenizer


In [ ]:
!zip tokenizers_2.zip tokenizers

  adding: tokenizers/ (stored 0%)


### Fertility score calucation for sangrah tokenizer

In [7]:
path = "sangrah_tokenizers\sangrah.csv_SentencePieceBPETokenizer_50000_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [8]:
# Tokenize a text segment
df = pd.read_csv("sangrah.csv")

art = df["text"]

In [9]:
len(tokenizer.encode(art.iloc[1])), len(art.iloc[1].split(" "))

(497, 435)

In [10]:
art.iloc[1]

"सर्लाही  -  चार दिनदेखि पूर्व-पश्चिम राजमार्गमा पर्ने इन्दु शंकर चिनी उद्योगको अघिल्तिर निरन्तर सवारी जाम छ । मिलमा उखु ल्याएका सवारी राख्ने ठाउँ नभएर राजमार्गमै अलपत्र छन् । क्षमताभन्दा बढी लोड भएका सवारी राजमार्गमा राखिएपछि सडक जाम भएको हो ।\n चिनी मिलले राख्ने ठाउँभन्दा बढी सवारी आउने अनुमति दिएपछि केही दिनदेखि राजमार्ग घण्टौ अवरुद्ध हुन थालेको छ । लगातार राजमार्ग अवरुद्ध भएपछि प्रहरी र मिलका कर्मचारीबीच भनाभन समेत भएको छ । मिलको लापरबाहीले गर्दा राजमार्ग अवरुद्ध भएको भन्दै प्रहरीले लोड गाडी राख्ने स्थान तय गरी मात्र उखु मगाउन आग्रह गरेको थियो ।\n दुई दिनदेखि प्रदेशका मुख्य सचिव र प्रहरी प्रमुखको यात्रा भइरहेकाले राजमार्गको जाम प्रहरीलाई टाउको दुखाइ बनेको छ । धुर्मस-सुन्तली फाउन्डेसनले रौतहटको चन्द्रपुरमा निर्माण गरेको बस्तीको मंगलबार हुने उद्घाटन कार्यक्रममा आउजाउ गर्ने विशिष्ट पाहुनालाई चिनी मिल अघिल्तिरको बाटो काट्नै मुस्किल पर्यो ।\n चिनी मिलले दैनिक आवश्यक पर्ने उखुका लागि मात्र ल्याउने अनुमति दिने गर्छ । चिनी मिलबीचको प्रतिस्पर्धाले बढी उखु मगाउने प्रचलन केही वर्षदेखि बढ्दै ग

In [11]:
# Convert list of encoding to nepali script
encoding  = tokenizer.encode(art.iloc[1])
text = ""
for i in range(len(encoding)):
  text += tokenizer.decode([encoding[i]])
print(text)
print(art.iloc[1])

सर्लाही  -  चार दिनदेखि पूर्व-पश्चिम राजमार्गमा पर्ने इन्दु शंकर चिनी उद्योगको अघिल्तिर निरन्तर सवारी जाम छ । मिलमा उखु ल्याएका सवारी राख्ने ठाउँ नभएर राजमार्गमै अलपत्र छन् । क्षमताभन्दा बढी लोड भएका सवारी राजमार्गमा राखिएपछि सडक जाम भएको हो ।
 चिनी मिलले राख्ने ठाउँभन्दा बढी सवारी आउने अनुमति दिएपछि केही दिनदेखि राजमार्ग घण्टौ अवरुद्ध हुन थालेको छ । लगातार राजमार्ग अवरुद्ध भएपछि प्रहरी र मिलका कर्मचारीबीच भनाभन समेत भएको छ । मिलको लापरबाहीले गर्दा राजमार्ग अवरुद्ध भएको भन्दै प्रहरीले लोड गाडी राख्ने स्थान तय गरी मात्र उखु मगाउन आग्रह गरेको थियो ।
 दुई दिनदेखि प्रदेशका मुख्य सचिव र प्रहरी प्रमुखको यात्रा भइरहेकाले राजमार्गको जाम प्रहरीलाई टाउको दुखाइ बनेको छ । धुर्मस-सुन्तली फाउन्डेसनले रौतहटको चन्द्रपुरमा निर्माण गरेको बस्तीको मंगलबार हुने उद्घाटन कार्यक्रममा आउजाउ गर्ने विशिष्ट पाहुनालाई चिनी मिल अघिल्तिरको बाटो काट्नै मुस्किल पर्यो ।
 चिनी मिलले दैनिक आवश्यक पर्ने उखुका लागि मात्र ल्याउने अनुमति दिने गर्छ । चिनी मिलबीचको प्रतिस्पर्धाले बढी उखु मगाउने प्रचलन केही वर्षदेखि बढ्दै गएको 

In [12]:
df =  df["text"].tolist()

Number of tokens

In [13]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

57923994


Number of words

In [14]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

46452863


In [15]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2469413133911682


#### Fertility score on another dataset

In [69]:
path = "./tokenizers/final_sample_40000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [71]:
# Tokenize a text segment
df = pd.read_csv("all_news_scraped_1.csv")

art = df["text"]

In [74]:
df =  df["text"].tolist()

Number of tokens

In [75]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

27166992


Number of words

In [76]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

21683125


In [77]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2529094399446574


### Fertility score calucation for 70000

In [ ]:
path = "./tokenizers/final_sample_70000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [ ]:
# Tokenize a text segment
df = pd.read_csv("final_sample_70000.csv")

art = df["text"]

In [ ]:
len(tokenizer.encode(art.iloc[1])), len(art.iloc[1].split(" "))

(550, 461)

In [ ]:
# Convert list of encoding to nepali script
encoding  = tokenizer.encode(art.iloc[1])
text = ""
for i in range(len(encoding)):
  text += tokenizer.decode([encoding[i]])
print(text)
print(art.iloc[1])

<bos>काफ्लेको यशोधाराको लोकार्पण

लेखनाथ   पोखरामा जारी नेपाल लिटरेचर फेस्टिभल २०२२ को दोस्रो दिन हरिबोल काफ्लेद्वारा लिखित पुस्तक यशोधराको लोकार्पण गएिको छ । सिद्धार्थ गौतम  बुद्ध  बन्ने यात्रामा यशोधराको भूमिका तथा त्यागको कथा रहेको उपन्यास आज लोकार्पण गरिएको हो ।
पुस्तकबारे बोल्दै समीक्षक तथा चिकित्सक जीवन क्षेत्रीले पुस्तकले गौतम बुद्धका बारेमा भनिँदै र सुनिँदै आएका कथामा खाली रहेका ठाउँहरू भरेको बताए। उनले भने,  यो पुस्तक हातमा परेपछि पहिलेका खाली ठाउँ भरिएला भन्ने अपेक्षा थियो । जसमा लेखक सफल पनि हुनुभएको छ । सधैँ गौतम बुद्धको कोणबाट सुन्दै पढ्दै आएको कथा लेखकले यशोधराको कोणबाट लेख्नुभएको छ । 
गौतम बुद्धसँगै पुस्तकले दुई हजार पाँच सय देखि तीन हजार वर्ष अघिको सामाजिक अवस्था तथा द्वन्द्वको पनि चित्रण गर्ने उनले बताए। पुस्तकमा सिद्धार्थ गौतम र यशोधराले सँगै बिताएको समयमा केन्द्रित भएर यशोधराको कोणबाट यो पुस्तक लेखिएको छ । उनले भने,  यशोधरा र सिद्धार्थ बीचको प्रेम, साथमा बिताएको १४ वर्ष, दुईजना साथसाथै हुँदा पनि रहेको मनोवैज्ञानिक दूरी, सामान्य मानवीय सम्बन्ध नहुँदाको पीडा, दुईबाट ती

In [ ]:
df =  df["text"].tolist()

Number of tokens

In [ ]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

29608367


Number of words

In [ ]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

22859695


In [ ]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2952214366814605


#### Fertility score for another dataset

In [78]:
path = "./tokenizers/final_sample_70000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [ ]:
# Tokenize a text segment
df = pd.read_csv("all_news_scraped_1.csv")

art = df["text"]

In [ ]:
df =  df["text"].tolist()

Number of tokens

In [79]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

27137824


Number of words

In [80]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

21683125


In [81]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2515642463897616


### Fertility score calucation for 100000

In [41]:
path = "./tokenizers/final_sample_100000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [42]:
# Tokenize a text segment
df = pd.read_csv("final_sample_100000.csv")

art = df["text"]

In [43]:
len(tokenizer.encode(art.iloc[1])), len(art.iloc[1].split(" "))

(543, 461)

In [44]:
# Convert list of encoding to nepali script
encoding  = tokenizer.encode(art.iloc[1])
text = ""
for i in range(len(encoding)):
  text += tokenizer.decode([encoding[i]])
print(text)
print(art.iloc[1])

<bos>काफ्लेको यशोधाराको लोकार्पण

लेखनाथ   पोखरामा जारी नेपाल लिटरेचर फेस्टिभल २०२२ को दोस्रो दिन हरिबोल काफ्लेद्वारा लिखित पुस्तक यशोधराको लोकार्पण गएिको छ । सिद्धार्थ गौतम  बुद्ध  बन्ने यात्रामा यशोधराको भूमिका तथा त्यागको कथा रहेको उपन्यास आज लोकार्पण गरिएको हो ।
पुस्तकबारे बोल्दै समीक्षक तथा चिकित्सक जीवन क्षेत्रीले पुस्तकले गौतम बुद्धका बारेमा भनिँदै र सुनिँदै आएका कथामा खाली रहेका ठाउँहरू भरेको बताए। उनले भने,  यो पुस्तक हातमा परेपछि पहिलेका खाली ठाउँ भरिएला भन्ने अपेक्षा थियो । जसमा लेखक सफल पनि हुनुभएको छ । सधैँ गौतम बुद्धको कोणबाट सुन्दै पढ्दै आएको कथा लेखकले यशोधराको कोणबाट लेख्नुभएको छ । 
गौतम बुद्धसँगै पुस्तकले दुई हजार पाँच सय देखि तीन हजार वर्ष अघिको सामाजिक अवस्था तथा द्वन्द्वको पनि चित्रण गर्ने उनले बताए। पुस्तकमा सिद्धार्थ गौतम र यशोधराले सँगै बिताएको समयमा केन्द्रित भएर यशोधराको कोणबाट यो पुस्तक लेखिएको छ । उनले भने,  यशोधरा र सिद्धार्थ बीचको प्रेम, साथमा बिताएको १४ वर्ष, दुईजना साथसाथै हुँदा पनि रहेको मनोवैज्ञानिक दूरी, सामान्य मानवीय सम्बन्ध नहुँदाको पीडा, दुईबाट ती

In [45]:
df =  df["text"].tolist()

Number of tokens

In [46]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

42080320


Number of words

In [47]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

32488158


In [48]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2952510265432715


#### Fertility score for another dataset

In [82]:
path = "./tokenizers/final_sample_100000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [ ]:
# Tokenize a text segment
df = pd.read_csv("all_news_scraped_1.csv")

art = df["text"]

In [ ]:
df =  df["text"].tolist()

Number of tokens

In [83]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

27123726


Number of words

In [84]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

21683125


In [85]:
fertility_score = cnt2/cnt
print(fertility_score)

1.25091406335572


### Fertility score calucation for 150000

In [51]:
path = "./tokenizers/final_sample_150000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [52]:
# Tokenize a text segment
df = pd.read_csv("final_sample_150000.csv")

art = df["text"]

In [53]:
len(tokenizer.encode(art.iloc[1])), len(art.iloc[1].split(" "))

(549, 461)

In [54]:
# Convert list of encoding to nepali script
encoding  = tokenizer.encode(art.iloc[1])
text = ""
for i in range(len(encoding)):
  text += tokenizer.decode([encoding[i]])
print(text)
print(art.iloc[1])

<bos>काफ्लेको यशोधाराको लोकार्पण

लेखनाथ   पोखरामा जारी नेपाल लिटरेचर फेस्टिभल २०२२ को दोस्रो दिन हरिबोल काफ्लेद्वारा लिखित पुस्तक यशोधराको लोकार्पण गएिको छ । सिद्धार्थ गौतम  बुद्ध  बन्ने यात्रामा यशोधराको भूमिका तथा त्यागको कथा रहेको उपन्यास आज लोकार्पण गरिएको हो ।
पुस्तकबारे बोल्दै समीक्षक तथा चिकित्सक जीवन क्षेत्रीले पुस्तकले गौतम बुद्धका बारेमा भनिँदै र सुनिँदै आएका कथामा खाली रहेका ठाउँहरू भरेको बताए। उनले भने,  यो पुस्तक हातमा परेपछि पहिलेका खाली ठाउँ भरिएला भन्ने अपेक्षा थियो । जसमा लेखक सफल पनि हुनुभएको छ । सधैँ गौतम बुद्धको कोणबाट सुन्दै पढ्दै आएको कथा लेखकले यशोधराको कोणबाट लेख्नुभएको छ । 
गौतम बुद्धसँगै पुस्तकले दुई हजार पाँच सय देखि तीन हजार वर्ष अघिको सामाजिक अवस्था तथा द्वन्द्वको पनि चित्रण गर्ने उनले बताए। पुस्तकमा सिद्धार्थ गौतम र यशोधराले सँगै बिताएको समयमा केन्द्रित भएर यशोधराको कोणबाट यो पुस्तक लेखिएको छ । उनले भने,  यशोधरा र सिद्धार्थ बीचको प्रेम, साथमा बिताएको १४ वर्ष, दुईजना साथसाथै हुँदा पनि रहेको मनोवैज्ञानिक दूरी, सामान्य मानवीय सम्बन्ध नहुँदाको पीडा, दुईबाट ती

In [55]:
df =  df["text"].tolist()

Number of tokens

In [56]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

63282723


Number of words

In [57]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

48813180


In [58]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2964269691095724


#### Fertility score for another dataset

In [86]:
path = "./tokenizers/final_sample_150000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [ ]:
# Tokenize a text segment
df = pd.read_csv("all_news_scraped_1.csv")

art = df["text"]

In [ ]:
df =  df["text"].tolist()

Number of tokens

In [87]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

27114277


Number of words

In [88]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

21683125


In [89]:
fertility_score = cnt2/cnt
print(fertility_score)

1.250478286686075


### Fertility score calucation for 200000

In [92]:
path = "./tokenizers/final_sample_200000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [62]:
# Tokenize a text segment
df = pd.read_csv("final_sample_200000.csv")

art = df["text"]

In [63]:
len(tokenizer.encode(art.iloc[1])), len(art.iloc[1].split(" "))

(542, 461)

In [64]:
# Convert list of encoding to nepali script
encoding  = tokenizer.encode(art.iloc[1])
text = ""
for i in range(len(encoding)):
  text += tokenizer.decode([encoding[i]])
print(text)
print(art.iloc[1])

<bos>काफ्लेको यशोधाराको लोकार्पण

लेखनाथ   पोखरामा जारी नेपाल लिटरेचर फेस्टिभल २०२२ को दोस्रो दिन हरिबोल काफ्लेद्वारा लिखित पुस्तक यशोधराको लोकार्पण गएिको छ । सिद्धार्थ गौतम  बुद्ध  बन्ने यात्रामा यशोधराको भूमिका तथा त्यागको कथा रहेको उपन्यास आज लोकार्पण गरिएको हो ।
पुस्तकबारे बोल्दै समीक्षक तथा चिकित्सक जीवन क्षेत्रीले पुस्तकले गौतम बुद्धका बारेमा भनिँदै र सुनिँदै आएका कथामा खाली रहेका ठाउँहरू भरेको बताए। उनले भने,  यो पुस्तक हातमा परेपछि पहिलेका खाली ठाउँ भरिएला भन्ने अपेक्षा थियो । जसमा लेखक सफल पनि हुनुभएको छ । सधैँ गौतम बुद्धको कोणबाट सुन्दै पढ्दै आएको कथा लेखकले यशोधराको कोणबाट लेख्नुभएको छ । 
गौतम बुद्धसँगै पुस्तकले दुई हजार पाँच सय देखि तीन हजार वर्ष अघिको सामाजिक अवस्था तथा द्वन्द्वको पनि चित्रण गर्ने उनले बताए। पुस्तकमा सिद्धार्थ गौतम र यशोधराले सँगै बिताएको समयमा केन्द्रित भएर यशोधराको कोणबाट यो पुस्तक लेखिएको छ । उनले भने,  यशोधरा र सिद्धार्थ बीचको प्रेम, साथमा बिताएको १४ वर्ष, दुईजना साथसाथै हुँदा पनि रहेको मनोवैज्ञानिक दूरी, सामान्य मानवीय सम्बन्ध नहुँदाको पीडा, दुईबाट ती

In [65]:
df =  df["text"].tolist()

Number of tokens

In [66]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

84338820


Number of words

In [67]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

65075773


In [68]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2960094995721372


#### Fertility score for another dataset

In [93]:
path = "./tokenizers/final_sample_200000.csv_SentencePieceBPETokenizer_49152_transformer"

# Load the saved tokenizer for inferencing
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

In [ ]:
# Tokenize a text segment
df = pd.read_csv("all_news_scraped_1.csv")

art = df["text"]

In [ ]:
df =  df["text"].tolist()

Number of tokens

In [94]:
cnt2=0
for art in df:
    cnt2 += len(tokenizer.encode(art))
print(cnt2)

27114254


Number of words

In [95]:
cnt = 0
for art in df:
    cnt += len(art.split())
print(cnt)

21683125


In [ ]:
fertility_score = cnt2/cnt
print(fertility_score)

1.2515642463897616


# Merging CSVs

In [ ]:
import pandas as pd
import glob

# Step 1: Merge all CSV files into one DataFrame
file_paths = glob.glob('./*.csv')  # Update the path as needed
df = pd.read_csv(file_paths[0])
for i in range(1, len(file_paths)):
  print(file_paths[i])
  df = pd.concat([df, pd.read_csv(file_paths[i])], ignore_index=True)

# Step 2: Save the merged DataFrame as a single CSV file
df.to_csv('final_merged_dataset.csv', index=False)


./wiki.csv
./himalkhabar.csv
./onlinekhabar.csv
./news24nepal.csv


In [ ]:
# Step 3: Create samples of varying sizes while maintaining the distribution of the original dataset
def create_samples_preserving_distribution(df, sample_sizes, output_prefix):
    """
    Create samples of varying sizes from the DataFrame while preserving the distribution.

    Parameters:
    df (pd.DataFrame): The combined DataFrame.
    sample_sizes (list): List of sizes for the samples.
    output_prefix (str): Prefix for the output sample files.
    """
    for size in sample_sizes:
        # Create a sample that preserves the distribution of the original dataset
        sample = df.sample(n=size, random_state=42, weights=None)
        sample_filename = f'{output_prefix}_{size}.csv'
        sample.to_csv(sample_filename, index=False)
        print(f'Sample of size {size} saved to {sample_filename}')

In [ ]:

# Step 4: Define sample sizes and call the function
sample_sizes = [40000, 70000, 100000, 150000, 200000]  # Adjust sample sizes as needed
create_samples_preserving_distribution(df, sample_sizes, 'final')

Sample of size 40000 saved to final_sample_40000.csv
Sample of size 70000 saved to final_sample_70000.csv
Sample of size 100000 saved to final_sample_100000.csv
Sample of size 150000 saved to final_sample_150000.csv
Sample of size 200000 saved to final_sample_200000.csv


In [ ]:
!zip all_final_datasets.zip final_sample_40000.csv final_sample_70000.csv final_sample_100000.csv final_sample_150000.csv final_sample_200000.csv

  adding: final_sample_40000.csv (deflated 80%)
  adding: final_sample_70000.csv (deflated 80%)
  adding: final_sample_100000.csv (deflated 80%)
  adding: final_sample_150000.csv (deflated 80%)
  adding: final_sample_200000.csv (deflated 80%)


# Extracting data from a paraquet file

In [1]:
import pyarrow.parquet as pq
import pandas as pd

# Step 1: Read the Parquet file
parquet_file = "data-1.parquet"  # Replace with your file path
table = pq.read_table(parquet_file)

# Step 2: Convert the data to a pandas DataFrame
df = table.to_pandas()

# Step 3: Determine the approximate number of rows for 600 MB
# Estimate row size (in bytes) using a sample of the DataFrame
sample_size = min(1000, len(df))  # Use up to 1000 rows for estimation
sample_bytes = df.iloc[:sample_size].memory_usage(index=True, deep=True).sum()
row_size = sample_bytes / sample_size  # Average size per row
rows_for_600mb = int((600 * 1024 * 1024) / row_size)

# Step 4: Extract the required rows
df_subset = df.iloc[:rows_for_600mb]

# Step 5: Write to a CSV file
csv_file = "sangrah.csv"  # Specify the output CSV file path
df_subset.to_csv(csv_file, index=False)

print(f"CSV file with approximately 600 MB data saved to {csv_file}")


CSV file with approximately 600 MB data saved to sangrah.csv
